## 📘 MILP Benchmark Problem: *pb-market-split8-70-4* (MIPLIB)

The **pb-market-split8-70-4** problem is a classic combinatorial instance from the **MIPLIB** benchmark collection. It belongs to a family of problems derived from **market splitting**, a type of discrete optimization that arises in network design, market allocation, and game-theoretic resource division.

This specific instance is categorized as a **pure binary programming problem**, meaning:

- All decision variables are **binary** (\(x_j \in \{0,1\}\))
- No general integer or continuous variables are present
- The objective and constraints are all **linear**

Pure binary MILPs are important in optimization because they capture:
- yes/no decisions
- selection problems
- partitioning or allocation structures
- combinatorial configurations

🔗 Problem page:  
https://miplib.zib.de/instance_details_pb-market-split8-70-4.html

---

### 🧮 Problem Structure

From the MIPLIB statistics for this instance:

- **Decision variables:** 560  
- **Binary variables:** 560  
- **General integer variables:** 0  
- **Continuous variables:** 0  
- **Constraints:** 602  
- **Coefficients nonzeros:** 10934  
- **Constraint types:** linear

This means the problem has 560 binary variables and 602 linear constraints, making it a **large pure binary linear program**.

The dense structure of this instance (large count of nonzeros) reflects significant interaction between variables—typical for market splitting and allocation problems.

---

### 🧠 Mathematical Formulation (abstract)

Although the full mathematical model is encoded in the MPS file, the abstract structure is:

$$
\begin{aligned}
\min\quad & c^\top x \\
\text{s.t.}\quad & A x \le b, \\
& x_j \in \{0,1\} \quad \forall j = 1,\ldots,560,
\end{aligned}
$$

where:
- \(x\) is the binary decision vector
- \(A x \le b\) represents the linear constraints governing feasibility
- \(c^\top x\) is a linear cost function to be minimized

Because every variable is binary, this is a **0/1 integer program**.

---

### 🧩 Why This Instance Is Interesting

The *pb-market-split8-70-4* problem is worth including in benchmark studies because:

- It is entirely **binary** — ideal for exploring combinatorial solver behavior.
- It is **large** relative to other pure binary MILPs, providing a challenge even for commercial solvers.
- It typically requires **significant branch-and-bound exploration** (unless presolved away), making it great for callback analysis and convergence visualization.
- Its structure (market splitting) is representative of applications in economics, network design, and discrete allocation.

---

### ⚙️ Solver Behavior (to be filled after running)

When solved in this notebook with Gurobi, you may observe:

- A potentially **large integrality gap** between the root LP relaxations and integer solutions, since binary models often have weak LP relaxations.
- A series of **heuristic improvements** early in the search.
- Bound tightening as the branch-and-bound tree grows.
- A measurable sequence of incumbent improvements over time and nodes (useful for plotting).

Describe these observations in the cells below after executing the solver.

---

### 🗂 What You Will Analyze

In this notebook we demonstrate:

- How to **load an MPS file** directly into Gurobi
- How to use **callbacks** to track solver progress
- How to **visualize objective convergence** over time and nodes
- How to interpret solver logs and solution statistics
- How large pure binary MILPs differ from mixed ILPs

---

### 📌 References

This instance comes from the **MIPLIB** library, a standard benchmark suite for mixed integer programming problems. MIPLIB collects real and synthetic instances that challenge modern solvers and are widely used for research and performance evaluation.

(Link: https://miplib.zib.de/)


In [ ]:
!pip -q install gurobipy

import os
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import numpy as np

mps_path = "/content/binary_pb-market-split8-70-4.mps"
if not os.path.exists(mps_path):
    raise FileNotFoundError(f"Not found: {mps_path}")

m = gp.read(mps_path)
m.Params.OutputFlag = 1

# --- storage ---
times = []
nodes = []
incumbents = []
bounds = []
events = []

def cb(model, where):
    # Record new feasible solutions (incumbent improvements)
    if where == GRB.Callback.MIPSOL:
        t = model.cbGet(GRB.Callback.RUNTIME)
        n = model.cbGet(GRB.Callback.MIPSOL_NODCNT)
        obj = model.cbGet(GRB.Callback.MIPSOL_OBJ)
        bnd = model.cbGet(GRB.Callback.MIPSOL_OBJBND)

        times.append(t); nodes.append(n)
        incumbents.append(obj); bounds.append(bnd)
        events.append("MIPSOL")

    # Record periodic progress (bound updates)
    elif where == GRB.Callback.MIP:
        t = model.cbGet(GRB.Callback.RUNTIME)
        n = model.cbGet(GRB.Callback.MIP_NODCNT)
        obj_best = model.cbGet(GRB.Callback.MIP_OBJBST)
        obj_bnd  = model.cbGet(GRB.Callback.MIP_OBJBND)

        if obj_best < GRB.INFINITY:
            times.append(t); nodes.append(n)
            incumbents.append(obj_best); bounds.append(obj_bnd)
            events.append("MIP")

# Run once with callback
m.optimize(cb)

print("Collected points:", len(times))
print("Status:", m.Status)
if m.SolCount > 0:
    print("Objective:", m.ObjVal)
    m.write("/content/solution.sol")
    m.write("/content/model_parsed.lp")
    print("Wrote: /content/solution.sol and /content/model_parsed.lp")

# ---- Plot: Objective vs time ----
plt.figure(figsize=(8,5))
plt.plot(times, np.log(incumbents), marker="o", label="Incumbent")
plt.plot(times, np.log(bounds), linestyle="--", label="Best bound")
plt.yscale("log")
plt.xlabel("Time (seconds)")
plt.ylabel("Objective value (log scale)")
plt.title("MILP Progress vs Time")
plt.grid(True)
plt.legend()
plt.show()

# ---- Plot: Objective vs nodes ----
plt.figure(figsize=(8,5))
plt.plot(nodes, incumbents, label="Incumbent")
plt.plot(nodes, bounds, linestyle="--", label="Best bound")

plt.yscale("log")
plt.xlabel("Branch-and-bound nodes")
plt.ylabel("Objective value (log scale)")
plt.title("MILP Objective Convergence vs Nodes")
plt.grid(True, which="both", linestyle="--", alpha=0.6)
plt.legend()
plt.show()

# As the search for the optimum advnaces, the integrality gap minimizes.

# Convert to numpy arrays
t = np.asarray(times, dtype=float)
inc = np.asarray(incumbents, dtype=float)
bnd = np.asarray(bounds, dtype=float)

# Safety: keep only finite entries
mask = np.isfinite(t) & np.isfinite(inc) & np.isfinite(bnd)
t, inc, bnd = t[mask], inc[mask], bnd[mask]

# Relative gap for minimization
eps = 1e-12
gap = (inc - bnd) / np.maximum(np.abs(inc), eps)

# Clip to avoid log(0) and negative values due to numerical noise
gap = np.maximum(gap, 1e-12)

# Optional: sort by time (sometimes callback gives non-monotone due to mixed events)
order = np.argsort(t)
t, gap = t[order], gap[order]

plt.figure(figsize=(8,5))
plt.plot(t, gap, marker="o", linewidth=1)
plt.yscale("log")
plt.xlabel("Time (seconds)")
plt.ylabel("Relative MIP gap (log scale)")
plt.title("MIP Gap vs Time")
plt.grid(True, which="both", linestyle="--", alpha=0.6)
plt.show()

# plot for Node count vs Best bound (and optional incumbent)
# This plot shows how the lower bound tightens as branch-and-bound explores nodes

n = np.asarray(nodes, dtype=float)
inc = np.asarray(incumbents, dtype=float)
bnd = np.asarray(bounds, dtype=float)

mask = np.isfinite(n) & np.isfinite(inc) & np.isfinite(bnd)
n, inc, bnd = n[mask], inc[mask], bnd[mask]

# Sort by node count
order = np.argsort(n)
n, inc, bnd = n[order], inc[order], bnd[order]

plt.figure(figsize=(8,5))

# Best bound vs nodes (main request)
plt.plot(n, bnd, linestyle="--", linewidth=2, label="Best bound")

# Optional: include incumbent for context
plt.plot(n, inc, linewidth=1, label="Incumbent")

plt.xlabel("Branch-and-bound nodes")
plt.ylabel("Objective value")
plt.title("Bound Tightening vs Node Count")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()


Read MPS format model from file /content/binary_pb-market-split8-70-4.mps
Reading time = 0.00 seconds
pb-market-split8-70-4: 17 rows, 71 columns, 1113 nonzeros
Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 17 rows, 71 columns and 1113 nonzeros (Min)
Model fingerprint: 0xd67b67b7
Model has 1 linear objective coefficients
Variable types: 0 continuous, 71 integer (71 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+03, 2e+03]
Presolve removed 9 rows and 1 columns
Presolve time: 0.01s
Presolved: 8 rows, 70 columns, 551 nonzeros
Variable types: 0 continuous, 70 integer (70 binary)

Root relaxation: objective 0.000000e+00, 21 iterations, 0.00 